# Chapter 02-03 · The rows you never see

**Label:** Core  |  **Time:** ~50 minutes  |  **Difficulty:** gentle to read, permanently useful

**Prerequisites:** 02-02. You should be able to ask the six provenance questions.

**Position in the learning path:** module 02, chapter 3 of 8. Before: **02-02**. After: **02-04**,
on the defects you *can* see.

---

## Why this matters

Provenance question 6 was **who or what is missing entirely?** - and it is the only one of the six
you cannot answer by studying the file, because the evidence is precisely what is not in it.

This is the failure mode with no symptoms. Missing values leave holes you can count. Wrong dtypes
raise errors. Rows that were never created leave nothing at all: the dataset is complete, tidy,
internally consistent, and about a different population from the one you care about.

In this chapter, 6% of Maria's customers are unhappy and her survey says 3%. Nobody lied, no row is
wrong, and the number that reaches her is half the truth.

## What you will be able to do

By the end of this chapter you can:

1. **Distinguish** population, sampling frame and sample, and name the gap between each pair.
2. **Recognise** the five common selection mechanisms and the direction each one biases a result.
3. **Quantify** response bias when the population is known, and reason about it when it is not.
4. **Diagnose** censoring - the observation that has not finished yet - and say why it shortens
   every duration you compute.
5. **Ask** the "who is missing?" questions of any dataset before trusting a summary of it.

## Warm-up: retrieve, do not reread

From memory:

1. What are the six provenance questions?
2. What was the diagnostic that caught the March surge?
3. Why is data from a previous model the most dangerous source?
4. What does a data dictionary have to record about missing values?

<br>

*Answers: (1) who/what created it; why; what each field means; when recorded relative to the event;
what changed over time; who is missing. (2) a slice where the real quantity cannot have changed -
undockings at 3am. (3) it records that model's decisions, not the world, so a new model learns to
imitate the old one. (4) the convention - whether an absent row means zero or means no measurement.*

## The situation

Maria wants to know how satisfied her customers are, so she puts a card on the counter: *rate us
from 1 to 5*. Over a month, about 550 cards come back with an average of **3.7**.

She has 2,000 customers.

The 549 who answered are not a random 549. They are the ones who were at the counter rather than at
the rack, who had a spare minute, who felt something worth writing down - and, as we will see, who
were more likely to be pleased.

**The question this chapter answers:** what is the difference between the people you measured and
the people you care about, and which way does it push your answer?

## Three groups, and the gaps between them

| | What it is | Maria's case |
|---|---|---|
| **Population** | Everyone the conclusion is about | All 2,000 customers |
| **Sampling frame** | Everyone who *could* have been sampled | Customers who came to the counter during the month |
| **Sample** | Everyone who actually appears in the data | The 549 who filled in a card |

Two gaps, and each has a name:

- **Population -> frame: a coverage gap.** People who used the app and never came to the counter had
  no chance of being surveyed. No amount of collecting more cards reaches them.
- **Frame -> sample: non-response.** Of those who could have answered, some did not - and *why* they
  did not is the whole question.

**The gap is only a problem when it is related to what you are measuring.** If the 549 differed from
the rest by height, nobody would care. They differ by satisfaction, which is the thing being
measured - and that is what makes a selection *biased* rather than merely *incomplete*.

### The five mechanisms

| Mechanism | What happens | Typical direction |
|---|---|---|
| **Coverage** | Part of the population cannot appear | Whatever that part is like, you learn nothing about |
| **Self-selection** | People choose whether to appear | Over-represents the motivated: the delighted and the furious |
| **Survivorship** | Only things that lasted are in the record | Over-states success - the failures were removed |
| **Censoring** | The event has not finished yet | Under-states duration - long ones are still running |
| **Selection on the outcome** | Inclusion depends on the thing you are studying | Anything. This is the most dangerous |

Survivorship is worth a sentence of its own because it is so common and so invisible. A table of
current customers contains no churned ones. A fund performance table contains no funds that closed.
A fleet table contains no bikes that were stolen. Each of those tables is complete, correct - and
about the survivors.

## The data

**SYNTHETIC**, and for once that is not a limitation but the entire method: we generate the whole
population, so we can see both what the survey said *and* what was true. In real life you only ever
have the left-hand column.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

rng = np.random.default_rng(21)
n_customers = 2000

satisfaction = np.clip(rng.normal(3.4, 1.0, n_customers), 1, 5).round(1)

# Happier customers are more likely to bother filling in a card.
response_chance = 0.03 + 0.40 * (satisfaction - 1) / 4
responded = np.random.default_rng(7).random(n_customers) < response_chance

customers = pd.DataFrame({"satisfaction": satisfaction, "responded": responded})
print(f"population    : {n_customers} customers")
print(f"responded     : {responded.sum()} cards ({responded.mean():.1%} response rate)")

### Predict before running

1. The true population mean is 3.41. Will the survey mean be higher, lower, or the same?
2. 7.4% of the population rate their experience 1 or 2. What percentage will the survey report?
3. Would collecting *twice as many cards* fix it?

In [ ]:
population_mean = customers["satisfaction"].mean()
survey_mean = customers.loc[customers["responded"], "satisfaction"].mean()

print(f"true population mean : {population_mean:.2f}")
print(f"survey mean          : {survey_mean:.2f}   ({survey_mean - population_mean:+.2f})")

In [ ]:
bands = pd.cut(customers["satisfaction"], [0.99, 2, 3, 4, 5], labels=["1-2", "2-3", "3-4", "4-5"])
comparison = pd.DataFrame({
    "population": bands.value_counts(normalize=True).sort_index(),
    "survey": bands[customers["responded"]].value_counts(normalize=True).sort_index(),
})
print((comparison * 100).round(1).to_string())

## Failure lab: the survey that halved the problem

**The mean is 3.73 against a true 3.41** - about a third of a star too high.

That is the number people quote, and it is not the worst of it. Look at the bottom band.

**7.4% of customers rate their experience 1 or 2. The survey reports 2.9%.** The share of unhappy
customers is understated by **more than half**.

That is the number Maria would act on. "Three per cent of customers are unhappy" is a rounding
error - you do nothing. "Seven and a half per cent" is one customer in thirteen, which at 2,000
customers is nearly 150 people, and it is a problem worth a morning.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
width = 0.38
positions = np.arange(len(comparison))
ax.bar(positions - width / 2, comparison["population"] * 100, width,
       color="#0072B2", label="all 2,000 customers (unobservable)")
ax.bar(positions + width / 2, comparison["survey"] * 100, width,
       color="#D55E00", label="the 549 who answered")
ax.set_xticks(positions, comparison.index)
ax.set_xlabel("Satisfaction rating"); ax.set_ylabel("Share of the group (%)")
ax.set_ylim(0, None)
ax.set_title("Who answered: the unhappy are half as likely to appear")
ax.legend(fontsize=8)
plt.show()

### Diagnosis

**Why it happened.** The chance of returning a card rose with satisfaction, from about 3% for the
most dissatisfied to about 43% for the most satisfied. Nobody was excluded; they were included at
different rates, and the rate depended on the very thing being measured. That is *selection on the
outcome*, the most dangerous row in the table above.

**Why more data does not help - and this is the point of the exercise.** Doubling the cards to 1,100
halves the *random* error and does nothing to the bias, because every extra card is drawn from the
same skewed process. The survey mean converges beautifully - to 3.73, which is the wrong number.

> **More data reduces variance. It does not reduce bias.** A biased estimate with a narrow
> confidence interval is a confident wrong answer, and looks more trustworthy than an honest one.

You met this in 00-04: with twenty million customers the naive email estimate converged to 20.42,
still four times the truth. It is the same statement.

**Which direction, and by how much?** The direction you can usually reason out: if responding is
easier for the satisfied, the survey overstates satisfaction. The *size* you generally cannot know -
here we know because we generated the population, which is exactly the luxury real work denies you.

### What to do when you cannot see the population

| Approach | What it does | What it costs |
|---|---|---|
| **Report the response rate next to every number** | Makes the gap visible - 27% response is a warning label | Nothing. Do it always |
| **Compare respondents with the frame on what you *do* know** | Age, plan, tenure, region are known for everyone. If respondents differ there, they differ elsewhere | One `groupby` |
| **Chase a random subsample of non-responders** | The only method that measures the bias rather than guessing | Expensive, and the gold standard |
| **Reweight by known characteristics** | Corrects for gaps you can measure | Cannot correct for gaps you cannot measure - the usual case |
| **Change the frame** | Survey at the rack and in the app, not only at the counter | Effort, and it fixes coverage rather than non-response |
| **Say the sentence** | "Among customers who responded..." | Nothing, and it prevents the wrong conclusion |

The last row is not a joke. **"Among the customers who answered, average satisfaction was 3.7"** is
true. **"Average customer satisfaction is 3.7"** is not. The difference is six words.

---

## Failure lab 2: the rentals that had not finished

A different mechanism, and the one that quietly shortens every duration anybody ever computes.

Maria wants the average rental length. She exports the log at 6pm and averages `duration_min`. But a
rental that started at 5:40pm and is still running has no duration yet - the row is either absent or
has an empty field, and it drops out of the average.

**Predict before running:** the true average rental is 20 minutes. What will the snapshot report -
higher, lower, or the same? Which rentals are missing?

In [ ]:
rng_dur = np.random.default_rng(4)
n_rentals = 3000

true_duration = rng_dur.exponential(20, n_rentals)          # minutes; SYNTHETIC
started_at = rng_dur.uniform(0, 120, n_rentals)             # minutes into a two-hour window
finished_before_snapshot = (started_at + true_duration) <= 120

print(f"true mean duration            : {true_duration.mean():.1f} min")
print(f"mean of completed rentals only: {true_duration[finished_before_snapshot].mean():.1f} min "
      f"({true_duration[finished_before_snapshot].mean() - true_duration.mean():+.1f})")
print(f"still out at the snapshot     : {(~finished_before_snapshot).mean():.1%} of rentals")

### Diagnosis

**19.9 minutes true, 16.0 minutes observed** - a fifth too short, from 16% of rentals being missing.

**Which rentals are missing is not random.** A rental is still running at the snapshot precisely
because it is *long*, or because it started late. Long rentals are systematically more likely to be
excluded, so the ones you can see are the short ones. This is **censoring**, and it has a direction
you can always predict: **any average of durations computed from completed cases is too short.**

It appears everywhere, usually unlabelled:

- **Time to churn**, computed only from customers who have already churned - the loyal ones have not
  churned *yet*, so they are excluded, and the estimate of customer lifetime is too short.
- **Time to failure** of a component, from the ones that failed. The reliable ones are still running.
- **Time to recovery**, from patients who recovered. Those still ill are excluded.
- **Time to conversion**, from customers who converted. The slow deciders are still deciding.

In every case the mechanism is the same and so is the direction. And in every case the naive fix -
"just wait longer" - only helps if you can wait longer than the longest case, which for customer
lifetime means waiting forever.

**What the right answer looks like.** The field that handles this properly is *survival analysis*,
which uses censored observations rather than discarding them: a rental still running at 40 minutes
tells you "this one is at least 40", which is real information. That is a genuine subject with real
prerequisites, and this course maps it honestly in 12-07 rather than pretending a notebook covers
it. What you need now is to **recognise** the shape - "am I averaging only the ones that finished?" -
and to say so.

**The cheap partial fix**, when survival analysis is out of scope: report the share censored
alongside the number, and restrict to a window in which almost everything completes. "Mean duration
16 minutes, but 16% of rentals were still running at the cut-off" is honest and usable. "Mean
duration 16 minutes" is not.

## The questions to ask of any dataset

Before believing any summary, run through these. They take two minutes.

1. **Who is in this table, and who had no chance of being in it?** (coverage)
2. **Did anyone choose whether to appear?** (self-selection)
3. **Is anything here because it survived something?** (survivorship)
4. **Has every row finished happening?** (censoring)
5. **Could inclusion depend on the thing I am measuring?** (selection on the outcome)
6. **What do I know about the people who are missing?** - and can I compare them on anything?

Question 6 is the practical one. You usually know *something* about non-responders - how long they
have been customers, which plan they are on, which region - because that comes from your own
records rather than from the survey. Comparing respondents with non-responders on those known fields
does not prove the bias is small, but a large difference proves it is not.

**And for ML specifically:** every one of these applies to the *training set*, and there is a
sharper version. If your model will be applied to people the training data could not include, you
are extrapolating - and the model has no way to tell you. A loan model trained on approved
applicants knows nothing about the rejected ones, because their outcomes were never observed. That
is 00-04's hospital-triage problem and 13-01's fairness problem, arriving from the same direction.

## Common misconceptions

**"A big sample is a good sample."**
Size fixes noise, not bias. A million self-selected reviews estimate the wrong number very
precisely. The famous 1936 *Literary Digest* poll had 2.4 million responses and called the
presidential election wrong; a poll of a few thousand, sampled properly, got it right.

**"Random means whatever I happened to collect."**
Random means every member of the population had a known, non-zero chance of being included. Cards on
a counter are not random; they are convenient.

**"I can fix it by weighting."**
You can correct for differences you can *measure*. If respondents and non-responders differ on
satisfaction itself - which is unmeasured for non-responders, by definition - no weighting scheme
recovers it.

**"Missing rows show up as missing values."**
They do not. A missing *value* is a hole in a row that exists. A missing *row* leaves nothing. This
is why `.isna().sum()` cannot find selection bias, and why the next chapter, about defects you can
see, is a different chapter.

**"Survivorship bias is a stock-market thing."**
It is in every table of current things: current employees, active users, in-service equipment, open
accounts. If the table is called "current" or "active", ask what left.

**"If I cannot measure the bias, I should not mention it."**
The opposite. An unquantified, named limitation is far more useful than silence: it tells the reader
which direction the number is wrong in, which is often enough to change a decision.

---

## Exercises

Solutions: `solutions/02_data_literacy/02-03_sampling_bias_solutions.ipynb`.

### Quick understanding

**E1 (define).** Define population, sampling frame and sample, and name the gap between each pair.

**E2 (explain).** Why does doubling the number of survey cards not reduce the bias? What does it
reduce?

**E3 (explain).** Why can't `.isna().sum()` detect selection bias?

### Hand calculation

**E4 (calculate).** A population of 1,000 customers: 700 satisfied (rating 4) and 300 dissatisfied
(rating 2). Satisfied customers respond with probability 0.30, dissatisfied with probability 0.10.
Compute: the number of responses expected from each group, the survey's reported mean, the true
mean, and the bias. Then recompute with response probabilities 0.60 and 0.20 - three times higher
for both. What happens to the bias, and why?

**E5 (calculate).** Of 3,000 rentals, 16% were still running at the 6pm snapshot. The completed ones
average 16 minutes. If every unfinished rental had lasted exactly 60 minutes, what would the true
mean have been? What does that tell you about how much the censoring could be hiding?

### Coding

**E6 (code).** Write `response_bias(population, responded, column)` that reports the population
mean, the respondent mean, the bias, the response rate, and the response rate within each quartile
of `column`. Run it on the chapter's data.

**E7 (code).** Simulate the *Literary Digest* effect: draw a sample of 50 with a *fair* random
process and a sample of 1,000 with the chapter's biased process. Compare each mean against the truth
over 500 repetitions, and plot the two distributions. Which sample would you rather have?

### Interpretation

**E8 (interpret).** A product team reports "our app store rating is 4.6, so users love it". Give
three selection mechanisms that could produce a 4.6 from a mediocre product, and one piece of data
you would ask for.

### Debugging

**E9 (diagnose).** A model predicting which job applicants will succeed is trained on the
performance reviews of people who were hired. It scores well in cross-validation and, once deployed,
recommends candidates who do not work out. Explain the selection mechanism, why cross-validation
could not catch it, and what would be needed to fix it properly.

### Exam and interview reasoning

**E10 (defend).** *"We have every transaction in the company - this is population data, not a
sample, so selection bias doesn't apply."* Reply in four sentences.

**E11 (design).** Maria wants a satisfaction number she can trust. Design the collection process:
where, when, whom, and how you would check afterwards that it worked. State the one thing your
design still cannot fix.

### Transfer to a different situation

**E12 (design).** For each, name the selection mechanism and the direction of the bias:
(a) estimating average customer lifetime from customers who have cancelled;
(b) estimating the effectiveness of a treatment from patients who completed the course;
(c) estimating the salary of graduates from an alumni survey;
(d) estimating fraud rates from transactions that were flagged and reviewed;
(e) estimating how long code reviews take from reviews that were finished.

### Explain it to someone non-technical

**E13 (explain).** In under 70 words, explain to Maria why her 3.7 is probably too high, without
using the words "bias", "sample" or "population".

### Optional challenge

**E14 (code + diagnose).** Show that reweighting works when the selection depends on something you
measured and fails when it depends on something you did not. Build two populations: in one,
responding depends on a recorded field (`plan`); in the other, on satisfaction itself. Apply
inverse-probability weighting using `plan` in both cases, and report the corrected means. Explain
the difference.

In [ ]:
# Your workspace. Still in memory: customers, satisfaction, responded, comparison,
# true_duration, started_at, finished_before_snapshot.

## Mastery check

Without scrolling up, can you:

- [ ] Name the two gaps between population, frame and sample? *(If not: "Three groups".)*
- [ ] List the five selection mechanisms? *(If not: "The five mechanisms".)*
- [ ] Say why more data does not fix bias? *(If not: "Failure lab" diagnosis.)*
- [ ] Predict the direction of censoring on any duration? *(If not: "Failure lab 2".)*
- [ ] Name the six-word fix for a survey headline? *(If not: the remedies table.)*

## What should now feel instinctive

1. **"Who had no chance of being in this table?"** - asked before any summary is believed.
2. **"Did anyone choose to be here?"** - and if so, what did they have in common.
3. **"Has every row finished happening?"** - durations from completed cases are always too short.
4. **More data narrows the interval around the wrong number.** Size is never the answer to bias.
5. **Say "among those who responded".** Six words, and the sentence becomes true.

## Flashcards

| Question | Answer |
|---|---|
| Population, frame, sample | Everyone the conclusion is about; everyone who could be sampled; everyone who appears |
| The two gaps | Coverage (population to frame) and non-response (frame to sample) |
| When is a gap a *bias*? | When it is related to the thing being measured |
| The five mechanisms | Coverage, self-selection, survivorship, censoring, selection on the outcome |
| Does more data fix selection bias? | No. It reduces variance, not bias - a confident wrong answer |
| Direction of censoring | Always understates duration: the long cases are still running |
| Why can't `.isna()` find it? | A missing row leaves nothing behind; only missing values leave holes |
| What does reweighting fix? | Differences on variables you measured. Never the ones you did not |
| The strongest fix for non-response | Chase a random subsample of non-responders and measure the gap |
| The cheapest honest fix | "Among those who responded..." |

## Next

**02-04 · Missing values, duplicates, impossible values and messy categories.**

This chapter was about defects that leave no trace. The next is about the ones that do - and the
good news is that they can be counted, which means they can be handled deliberately. The bad news is
that the most common handling ("drop the rows with missing values") is itself a selection mechanism,
and you have just spent a chapter learning why that matters.

New terms are in [GLOSSARY.md](../../GLOSSARY.md).